# 08 · Validación de la recuperación y depuración del grafo

**Proyecto:** Memoria Chilena — GraphRAG  
**Entrada:** `data/output/etapa7.zip` (generado en la etapa 7)  
**Salida:** `data/output/etapa8.zip` (automática)

Esta etapa **no descarga PDF, no hace OCR y no llama a ningún LLM**. Su finalidad es corregir el efecto de la etiqueta temática común, hacer la búsqueda más interpretable y preparar una evaluación humana sin inventar etiquetas de relevancia.

> Advertencia metodológica: los vínculos son **relaciones curatoriales**. No indican influencia histórica ni conexión causal. Las coincidencias de metadatos no demuestran el contenido de una obra.

In [1]:
from pathlib import Path
from collections import Counter
import json, zipfile, re, unicodedata, math
from datetime import datetime, timezone
import numpy as np
import pandas as pd
import networkx as nx
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity

# Funciona desde notebooks/ (Jupyter) o desde la raíz del proyecto.
cwd = Path.cwd().resolve()
if (cwd / 'data').exists() and (cwd / 'notebooks').exists():
    ROOT = cwd
elif cwd.name == 'notebooks' and (cwd.parent / 'data').exists():
    ROOT = cwd.parent
elif (cwd.parent / 'data').exists() and (cwd.parent / 'notebooks').exists():
    ROOT = cwd.parent
else:
    raise FileNotFoundError('Abre este notebook desde la carpeta notebooks/ del proyecto memoria_chilena.')

ENTRADA = ROOT / 'data' / 'output' / 'etapa7.zip'
if not ENTRADA.is_file():
    alternativa = ROOT / 'data' / 'input' / 'etapa7.zip'
    ENTRADA = alternativa if alternativa.is_file() else ENTRADA
assert ENTRADA.is_file(), f'Falta {ENTRADA}: ejecuta 07_Recuperacion_Palabras_Clave.ipynb primero.'
SALIDA = ROOT / 'data' / 'output' / 'etapa8'
SALIDA.mkdir(parents=True, exist_ok=True)
print('Proyecto:', ROOT)
print('Entrada:', ENTRADA)
print('Salida:', SALIDA)

Proyecto: C:\Users\jvera\Documents\memoria_chilena
Entrada: C:\Users\jvera\Documents\memoria_chilena\data\output\etapa7.zip
Salida: C:\Users\jvera\Documents\memoria_chilena\data\output\etapa8


In [2]:
import io
with zipfile.ZipFile(ENTRADA) as z:
    archivos = set(z.namelist())
    obligatorios = {'catalogo_para_busqueda.csv', 'relaciones_catalograficas.csv', 'palabras_clave.csv'}
    faltan = obligatorios - archivos
    assert not faltan, f'Faltan archivos en etapa7.zip: {faltan}'
    catalogo = pd.read_csv(io.BytesIO(z.read('catalogo_para_busqueda.csv'))).fillna('')
    relaciones = pd.read_csv(io.BytesIO(z.read('relaciones_catalograficas.csv'))).fillna('')
    palabras = pd.read_csv(io.BytesIO(z.read('palabras_clave.csv'))).fillna('')
    informe7 = json.loads(z.read('informe_etapa7.json').decode('utf-8')) if 'informe_etapa7.json' in archivos else {}

assert catalogo['mcid'].is_unique, 'Hay MCID duplicados en catálogo'
assert len(catalogo) == 10, f'La etapa 7 contiene {len(catalogo)} fichas, no las 10 del piloto'
print(f'Fichas: {len(catalogo)} | Palabras clave: {len(palabras)} | Pares catalográficos: {len(relaciones)}')

Fichas: 10 | Palabras clave: 114 | Pares catalográficos: 45


## 1. Auditar fechas y metadatos

Las fechas del RDF pueden describir un rango cronológico o una clasificación del catálogo. No las convertimos automáticamente en años de publicación. Se conserva el valor original y se señala cualquier fecha que requiera revisión.

In [3]:
def parse_fecha(valor):
    s=str(valor).strip()
    m=re.fullmatch(r'start=(\d{1,4});end=(\d{1,4})',s)
    if m:
        a,b=map(int,m.groups())
        return a,b,'rango catalográfico (NO asumido como año de edición)'
    if re.fullmatch(r'\d{4}',s):
        a=int(s);return a,a,'año simple sin verificación adicional'
    return None,None,'sin año interpretable'

fechas=catalogo[['mcid','titulo','fecha']].copy()
parsed=fechas['fecha'].map(parse_fecha)
fechas['anio_inicio']=[x[0] for x in parsed]
fechas['anio_fin']=[x[1] for x in parsed]
fechas['interpretacion']=[x[2] for x in parsed]
fechas['revisar']=[bool(a is not None and (a<1400 or b>2100 or b<a)) for a,b,_ in parsed]
fechas.to_csv(SALIDA/'fechas_auditoria.csv',index=False,encoding='utf-8-sig')
print('Registros a revisar por rango temporal inusual:',fechas['revisar'].sum())
display(fechas[['mcid','fecha','interpretacion','revisar']])

Registros a revisar por rango temporal inusual: 1


,mcid,fecha,interpretacion,revisar
0,MC0014753,1584,año simple sin verificación adicional,False
1,MC0014754,1553,año simple sin verificación adicional,False
2,MC0014755,start=1609;end=1609,rango catalográfico (NO asumido como año de ed...,False
3,MC0014757,start=1590;end=1590,rango catalográfico (NO asumido como año de ed...,False
4,MC0014760,start=1621;end=1621,rango catalográfico (NO asumido como año de ed...,False
5,MC0014761,start=1612;end=1612,rango catalográfico (NO asumido como año de ed...,False
6,MC0023126,start=1899;end=1999,rango catalográfico (NO asumido como año de ed...,False
7,MC0023128,start=1800;end=1999,rango catalográfico (NO asumido como año de ed...,False
8,MC0038283,start=2003;end=2003,rango catalográfico (NO asumido como año de ed...,False
9,MC0038290,start=900;end=900,rango catalográfico (NO asumido como año de ed...,True


## 2. Depurar conexiones temáticas *solo para la búsqueda*

Toda la colección comparte `religiosidadandina`. La preservamos en los datos originales, pero no la usamos para expandir consultas, ya que conecta artificialmente todos los pares. Se eliminan de la expansión solamente las etiquetas que aparecen en el 100 % de las fichas; el umbral se deriva de los datos, no de un vocabulario fijo.

In [4]:
def separar_unidades(texto):
    return {u.strip() for u in str(texto).split('|') if u.strip()}

unidades_por_doc={r.mcid:separar_unidades(r.unidades) for r in catalogo.itertuples()}
frecuencia=Counter(u for us in unidades_por_doc.values() for u in us)
n=len(catalogo)
universales={u for u,c in frecuencia.items() if c==n}
frecuencias_df=pd.DataFrame([{'unidad':u,'n_documentos':c,'proporcion':c/n,'uso_expansion':'excluir (universal)' if u in universales else 'admitir'} for u,c in sorted(frecuencia.items(),key=lambda z:(-z[1],z[0]))])
frecuencias_df.to_csv(SALIDA/'unidades_frecuencias.csv',index=False,encoding='utf-8-sig')

G=nx.Graph()
for r in catalogo.itertuples():
    G.add_node(str(r.mcid), titulo=str(r.titulo), ficha=str(r.ficha))

aristas=[]
ids=sorted(unidades_por_doc)
for i,a in enumerate(ids):
    for b in ids[i+1:]:
        comunes=sorted((unidades_por_doc[a] & unidades_por_doc[b])-universales)
        if comunes:
            # Criterio interpretable: inverso a la frecuencia documental de la unidad
            peso=round(sum(math.log((n+1)/(frecuencia[u]+1)) for u in comunes),5)
            G.add_edge(a,b,weight=peso,unidades=' | '.join(comunes))
            aristas.append({'mcid_origen':a,'mcid_destino':b,'unidades_especificas':' | '.join(comunes),'n_unidades':len(comunes),'peso_idf':peso,'tipo':'co-membresía catalográfica; NO relación histórica'})

edges=pd.DataFrame(aristas,columns=['mcid_origen','mcid_destino','unidades_especificas','n_unidades','peso_idf','tipo'])
edges.to_csv(SALIDA/'relaciones_filtradas.csv',index=False,encoding='utf-8-sig')
nx.write_graphml(G,SALIDA/'grafo_catalografico_filtrado.graphml')
print('Unidades comunes a todo el corpus:',universales)
print(f'Aristas originales: {len(relaciones)} -> Aristas específicas: {G.number_of_edges()}')
print('Documentos aislados tras filtrar:',list(nx.isolates(G)))
display(edges.head(12))

Unidades comunes a todo el corpus: {'religiosidadandina'}
Aristas originales: 45 -> Aristas específicas: 14
Documentos aislados tras filtrar: ['MC0038283', 'MC0038290']


,mcid_origen,mcid_destino,unidades_especificas,n_unidades,peso_idf,tipo
0,MC0014753,MC0014755,elpuebloaymara,1,0.60614,co-membresía catalográfica; NO relación histórica
1,MC0014753,MC0014757,elpuebloaymara,1,0.60614,co-membresía catalográfica; NO relación histórica
2,MC0014753,MC0014760,elpuebloaymara,1,0.60614,co-membresía catalográfica; NO relación histórica
3,MC0014753,MC0014761,elpuebloaymara | lasmisionesjesuitasdeparaguay...,2,1.90542,co-membresía catalográfica; NO relación histórica
4,MC0014754,MC0014755,cronistasperuanos,1,0.78846,co-membresía catalográfica; NO relación histórica
5,MC0014754,MC0014757,cronistasperuanos,1,0.78846,co-membresía catalográfica; NO relación histórica
6,MC0014754,MC0014760,cronistasperuanos,1,0.78846,co-membresía catalográfica; NO relación histórica
7,MC0014755,MC0014757,cronistasperuanos | elpuebloaymara,2,1.39459,co-membresía catalográfica; NO relación histórica
8,MC0014755,MC0014760,cronistasperuanos | elpuebloaymara,2,1.39459,co-membresía catalográfica; NO relación histórica
9,MC0014755,MC0014761,elpuebloaymara,1,0.60614,co-membresía catalográfica; NO relación histórica


## 3. Prototipo de recuperación más claro

Se consulta título, autor, materias, notas y datos bibliográficos con TF-IDF de palabras y caracteres. Las unidades **universales** no se incluyen en el texto de búsqueda. Se aceptan algunas grafías equivalentes para facilitar la consulta (por ejemplo, *quechua/quichua*), sin reescribir la fuente original. El grafo específico sirve para sugerir documentos vecinos, pero no fabrica evidencias.

In [5]:
def normalizar(s):
    s=unicodedata.normalize('NFKD',str(s).lower())
    s=''.join(c for c in s if not unicodedata.combining(c))
    s=s.replace('%20',' ')
    s=re.sub(r'[^a-z0-9\s]',' ',s)
    return re.sub(r'\s+',' ',s).strip()

ALIASES={'quichua':'quechua','piru':'peru','yncas':'incas'}
def expandir_alias(s):
    s=normalizar(s)
    for a,b in ALIASES.items():
        s=re.sub(r'\b'+a+r'\b',b,s)
    return s

def texto_de_registro(row):
    partes=[]
    for col,rep in [('titulo',4),('autor',2),('materias',1),('notas',2),('datos',1)]:
        partes += [expandir_alias(row[col])]*rep
    unidades=' '.join(sorted(u for u in unidades_por_doc[row['mcid']] if u not in universales))
    partes += [expandir_alias(unidades)]*2
    return ' '.join(partes)

corpus=[texto_de_registro(r) for _,r in catalogo.iterrows()]
vec_pal=TfidfVectorizer(ngram_range=(1,2),sublinear_tf=True,token_pattern=r'(?u)\b\w\w+\b')
vec_car=TfidfVectorizer(analyzer='char',ngram_range=(3,5),sublinear_tf=True)
Xp=vec_pal.fit_transform(corpus);Xc=vec_car.fit_transform(corpus)
mcids=catalogo['mcid'].tolist(); pos={v:i for i,v in enumerate(mcids)}

def recuperar(consulta,top_k=5,umbral=0.035):
    q=expandir_alias(consulta)
    # Combinar dos señales léxicas. Los valores no son probabilidades de relevancia.
    sp=cosine_similarity(vec_pal.transform([q]),Xp)[0]
    sc=cosine_similarity(vec_car.transform([q]),Xc)[0]
    base=.7*sp+.3*sc
    candidatos=[]
    for j,docid in enumerate(mcids):
        if base[j]<umbral: continue
        r=catalogo.iloc[j]
        vecinos=[{'mcid':v,'unidades_comunes':G[docid][v]['unidades']} for v in G.neighbors(docid)]
        candidatos.append({'consulta':consulta,'mcid':docid,'score_lexico':round(float(base[j]),4),'titulo':r['titulo'],'ficha':r['ficha'],'vecinos_especificos':len(vecinos),'evidencia':'solo metadatos; ver fuente'})
    return pd.DataFrame(sorted(candidatos,key=lambda x:-x['score_lexico'])[:top_k],columns=['consulta','mcid','score_lexico','titulo','ficha','vecinos_especificos','evidencia'])

queries_demo=['catecismo y quechua','vocabulario aymara','extirpación de idolatría','historia de los incas','La Tirana Tarapacá']
resultados=pd.concat([recuperar(q) for q in queries_demo],ignore_index=True)
resultados.to_csv(SALIDA/'consultas_ejemplo.csv',index=False,encoding='utf-8-sig')
for q in queries_demo:
    print('\n',q)
    display(recuperar(q)[['mcid','score_lexico','titulo','vecinos_especificos']])


 catecismo y quechua


,mcid,score_lexico,titulo,vecinos_especificos
0,MC0014753,0.2066,Doctrina christiana y catecismo para la instru...,4



 vocabulario aymara


,mcid,score_lexico,titulo,vecinos_especificos
0,MC0014761,0.2374,Vocabulario de la lengua aymara,4
1,MC0014753,0.0679,Doctrina christiana y catecismo para la instru...,4



 extirpación de idolatría


,mcid,score_lexico,titulo,vecinos_especificos
0,MC0014760,0.3129,Extirpación de la idolatria del Piru,5
1,MC0014755,0.0800,"Primera parte de los commentarios reales, que ...",5



 historia de los incas


,mcid,score_lexico,titulo,vecinos_especificos
0,MC0014755,0.2041,"Primera parte de los commentarios reales, que ...",5
1,MC0014757,0.0986,Historia natural y moral de las Indias : en qu...,5
2,MC0014753,0.0948,Doctrina christiana y catecismo para la instru...,4
3,MC0014754,0.0936,Parte primera de la Chrónica del Perú : que tr...,3
4,MC0014761,0.0549,Vocabulario de la lengua aymara,4



 La Tirana Tarapacá


,mcid,score_lexico,titulo,vecinos_especificos
0,MC0023126,0.5293,La Tirana de Tarapacá,1


## 4. Pruebas básicas y evaluación humana

Las cinco consultas anteriores son **controles funcionales**, elegidos porque se conoce qué título debería aparecer primero. No constituyen un conjunto independiente de evaluación. Para medir efectividad real, hay que completar la columna `relevancia_0_1_2` en la plantilla que genera esta celda tras revisar las fichas y, cuando sea necesario, los documentos.

In [6]:
checks={
    'catecismo y quechua':'MC0014753',
    'vocabulario aymara':'MC0014761',
    'extirpación de idolatría':'MC0014760',
    'historia de los incas':'MC0014755',
    'La Tirana Tarapacá':'MC0023126',
}
comprobaciones=[]
for q,esperado in checks.items():
    rr=recuperar(q,top_k=10)
    rank=list(rr['mcid']).index(esperado)+1 if esperado in list(rr['mcid']) else None
    comprobaciones.append({'consulta':q,'mcid_esperado':esperado,'posicion':rank,'hit_1':int(rank==1)})
comprobaciones=pd.DataFrame(comprobaciones)
comprobaciones.to_csv(SALIDA/'controles_funcionales.csv',index=False,encoding='utf-8-sig')
print('Controles funcionales top1:',sum(comprobaciones.hit_1),'/',len(comprobaciones))
display(comprobaciones)

preguntas_evaluacion=[
    '¿Qué obras catalogadas tratan sobre el aymara?',
    '¿Qué textos coloniales están vinculados a la evangelización?',
    '¿Qué documentos relacionan misiones jesuitas y lenguas indígenas?',
    '¿Qué obras corresponden a cronistas del Perú?',
    '¿Qué documentos permiten estudiar la geografía cultural del norte de Chile?',
    '¿Qué fuentes se relacionan con la fiesta de La Tirana?',
    '¿Qué obras están catalogadas sobre bailes chinos?',
    '¿Qué materiales hay acerca de idolatría en el Virreinato?',
    '¿Qué obras se relacionan con el Tawantinsuyu?',
    '¿Qué fuentes están vinculadas con Historia natural y moral de las Indias?',
    '¿Qué obras están asociadas con el pueblo aymara?',
    '¿Qué documentos se relacionan con religiosidad andina en Chile?',
]
plantilla=[]
for q in preguntas_evaluacion:
    r=recuperar(q,top_k=5,umbral=0.0)
    for x in r.to_dict(orient='records'):
        plantilla.append({'pregunta':q,'mcid':x['mcid'],'titulo':x['titulo'],'ficha':x['ficha'],'posicion_propuesta':len([p for p in plantilla if p['pregunta']==q])+1,'score_propuesto':x['score_lexico'],'relevancia_0_1_2':'','justificacion':'','evaluador':''})
plantilla=pd.DataFrame(plantilla)
plantilla.to_csv(SALIDA/'PLANTILLA_EVALUACION_MANUAL.csv',index=False,encoding='utf-8-sig')
print('Plantilla generada:',len(preguntas_evaluacion),'preguntas x 5 documentos =',len(plantilla),'juicios potenciales')

Controles funcionales top1: 5 / 5


,consulta,mcid_esperado,posicion,hit_1
0,catecismo y quechua,MC0014753,1,1
1,vocabulario aymara,MC0014761,1,1
2,extirpación de idolatría,MC0014760,1,1
3,historia de los incas,MC0014755,1,1
4,La Tirana Tarapacá,MC0023126,1,1


Plantilla generada: 12 preguntas x 5 documentos = 60 juicios potenciales


## 5. Empaquetar resultados

Todos los resultados quedan en `data/output/etapa8/` y se comprimen automáticamente como `data/output/etapa8.zip`. No es necesario copiar ni mover archivos.

In [7]:
resumen={
 'etapa':8, 'generado_utc':datetime.now(timezone.utc).isoformat(),
 'fuente':'etapa7.zip', 'fichas':int(len(catalogo)),
 'triples_principales_declarados_etapa7':informe7.get('triples_corpus_principal'),
 'aristas_originales':int(len(relaciones)),
 'aristas_especificas_para_expansion':int(G.number_of_edges()),
 'unidades_universales_excluidas_de_expansion':sorted(universales),
 'documentos_aislados_tras_filtro':list(nx.isolates(G)),
 'n_consultas_control':len(checks),
 'control_top1_aciertos':int(comprobaciones['hit_1'].sum()),
 'n_preguntas_evaluacion_pendientes':len(preguntas_evaluacion),
 'n_juicios_evaluacion_pendientes':len(plantilla),
 'advertencias':[
     'Los resultados actuales provienen de metadatos: no demuestran contenido de obras.',
     'El grafo filtrado es de co-membresía catalográfica: no representa relaciones históricas.',
     'La evaluación humana está pendiente; no afirmar precisión/recall de recuperación.',
     'No se han utilizado OCR ni PDF en esta etapa.',
     'Los rangos de fecha del RDF no necesariamente expresan fechas editoriales.'
 ]
}
(SALIDA/'informe_etapa8.json').write_text(json.dumps(resumen,ensure_ascii=False,indent=2),encoding='utf-8')
ZIP=ROOT/'data'/'output'/'etapa8.zip'
with zipfile.ZipFile(ZIP,'w',zipfile.ZIP_DEFLATED) as z:
    for f in sorted(SALIDA.iterdir()):
        if f.is_file():z.write(f,f.name)
print('✅ Generado:',ZIP)
print('Contenido:')
with zipfile.ZipFile(ZIP) as z:print('\n'.join(' - '+n for n in z.namelist()))

✅ Generado: C:\Users\jvera\Documents\memoria_chilena\data\output\etapa8.zip
Contenido:
 - consultas_ejemplo.csv
 - controles_funcionales.csv
 - fechas_auditoria.csv
 - grafo_catalografico_filtrado.graphml
 - informe_etapa8.json
 - PLANTILLA_EVALUACION_MANUAL.csv
 - relaciones_filtradas.csv
 - unidades_frecuencias.csv
